## **2. SELECCIÓN Y ENTRENAMIENTO DE LOS MODELOS**
<hr>

### **2.1. Pipeline de Preprocesamiento**

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.base import BaseEstimator, TransformerMixin
from src.evaluation import auditar_modelo

# 1. PURGAMOS LA VARIABLES SELECCIONADAS DURANTE LA EDA Y AQUELLAS INNECESARIAS
COLS_DROP = ['label', 'D', 'comentario', 'A_t', 'B_t', 'C_t', 'D_t', 'E_t', 'Valor_1', 'Valor_2', 'Valor_5'] # Más adelante decidimos si eliminar Valor_6 o Valor_8
X = df.drop(columns=COLS_DROP)
y = df['label']

# 2. PARTICIONADO ESTRATIFICADO
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. TRANSFORMADOR: CLIPPING
class P99Clipper(BaseEstimator, TransformerMixin):
    """
    Acota la cola larga extrema de las distribuciones.
    Aprende los umbrales SOLO en .fit() para evitar Data Leakage.
    """
    def __init__(self, percentile=99.0):
        self.percentile = percentile
        self.upper_bounds_ = None

    def fit(self, X, y=None):
        # Cálculo sobre el conjunto de entrenamiento
        self.upper_bounds_ = np.percentile(X, self.percentile, axis=0)
        return self

    def transform(self, X, y=None):
        # Aplicación del límite
        return np.clip(X, a_min=None, a_max=self.upper_bounds_)

# 4. MAPEO DE CARACTERÍSTICAS
COLS_NUMERICAS = ['A', 'B', 'C', 'Valor_3', 'Valor_4', 'Valor_6', 'Valor_7', 'Valor_9', 'Valor_10'] # Optamos por eliminar la variable Valor_8
COLS_ORDINALES = ['E']

# 5. CONSTRUCCIÓN DEL PIPELINE
num_pipeline = Pipeline(steps=[
    ('clipper', P99Clipper(percentile=99.0)),
    ('scaler', RobustScaler())
])

preprocesador = ColumnTransformer(
    transformers=[
        ('num_transforms', num_pipeline, COLS_NUMERICAS),
        ('cat_ordinal', 'passthrough', COLS_ORDINALES)
    ],
    remainder='drop'
)

# 6. VALIDACIÓN DEL FLUJO
print(f"Dimensiones X_train crudo: {X_train.shape}")
X_train_procesado = preprocesador.fit_transform(X_train)
X_test_procesado = preprocesador.transform(X_test)
print(f"Dimensiones X_train procesado: {X_train_procesado.shape}")

Dimensiones X_train crudo: (7568, 11)
Dimensiones X_train procesado: (7568, 10)


In [20]:
df.to_csv('../data/processed/dataset_hate_speech_cleaned.csv', index=False)

### **2.2. Regresión Logística ElasticNet**

#### I. Entrenamiento del Baseline

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# 1. Definición del Modelo Baseline --> ElasticNet
modelo_base = LogisticRegression(
    solver='saga', # Único algoritmo que soporta 'elastiknet'
    l1_ratio=0.5, # 50% Lasso (fuerza esparsidad) y 50% Ridge (controla multicolinealidad residual)
    max_iter=2000, # Amplio para garantizar convergencia del gradiente
    random_state=42
)

# 2. Ensamble del Pipeline
pipeline_lr = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', modelo_base)
])

pipeline_lr

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_transforms', ...), ('cat_ordinal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diff

In [22]:
# 3. Validación Cruzada Estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 4. Definimos el diccionario con las métricas seleccionadas durante la EDA
metricas = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'f1_macro': 'f1_macro'
}

# 5. Ejecutamos la Validación Cruzada y auditamos los resultados
cv_results = auditar_modelo(
    pipeline=pipeline_lr,
    X=X_train,
    y=y_train,
    cv=skf,
    metricas=metricas,
    nombre_modelo="Baseline ElasticNet"
)

Entrenando Baseline ElasticNet con Validación Cruzada...

-------------------------------
MÉTRICAS DE VALIDACIÓN CRUZADA: BASELINE ELASTICNET
-------------------------------
- ROC-AUC  : 0.9944 (+/- 0.0022)
- Accuracy : 0.9738 (+/- 0.0029)
- F1-Macro : 0.9738 (+/- 0.0029)

--------------------------------------
AUDITORÍA DE SOBREAJUSTE (TRAIN vs VAL)
--------------------------------------
- Accuracy Gap : Train (0.9740) vs Val (0.9738) -> Diferencia: 0.0001
- ROC-AUC Gap  : Train (0.9949) vs Val (0.9944) -> Diferencia: 0.0005


**Evaluación del Baseline --> Regresión Logística ElasticNet**

Los resultados de la validación cruzada en el modelo base revelan un rendimiento excepcionalmente alto:

* **Discriminación algorítmica:** Un ROC-AUC de 0.9944 indica una separabilidad lineal cuasi-perfecta entre las clases.
* **Robustez:** El hecho de que Accuracy y F1‑Macro sean prácticamente iguales (0.9738) indica que el modelo funciona igual de bien para ambas clases. Además, la baja variabilidad entre folds confirma que el rendimiento es sólido y no depende de la partición de los datos.

**¡Advertencia!**: Un *baseline* paramétrico con un rendimiento superior al 99% es un indicio que merece revisarse con cuidado. Esto suele ocurrir en dos situaciones: o bien las variables explicativas están tan fuertemente relacionadas con la variable objetivo que el problema resulta casi trivial, o existe algún tipo de fuga de información (Target Leakage) que está facilitando al modelo un acceso indirecto a la respuesta correcta.

Antes de dar por válidas estas métricas o pasar a modelos más complejos, es importante revisar a fondo la Regresión Logística. Para ello, conviene analizar la magnitud de sus coeficientes y comprobar que ninguna variable esté actuando como un sustituto directo de la etiqueta, lo que podría indicar un problema de fuga de información.

#### II. Auditoría de coeficientes

In [23]:
# 1. Ajustamos el pipeline final con todo el conjunto de entrenamiento
pipeline_lr.fit(X_train, y_train)

# 2. Extraemos el estimador (el modelo matemático) de dentro del pipeline
modelo_entrenado = pipeline_lr.named_steps['modelo']

# 3. Extraemos los coeficientes (Pesos W)
pesos_w = modelo_entrenado.coef_[0]

# 4. Reconstruimos los nombres de las columnas (numéricas + ordinal)
nombres_columnas = COLS_NUMERICAS + COLS_ORDINALES

# 5. Creamos un DataFrame para auditar la importancia de forma estructurada
df_auditoria_pesos = pd.DataFrame({
    'Característica': nombres_columnas,
    'Peso_W': pesos_w,
    'Magnitud_Absoluta': np.abs(pesos_w)
})

# Ordenamos por la magnitud del impacto (de mayor a menor)
df_auditoria_pesos = df_auditoria_pesos.sort_values(by='Magnitud_Absoluta', ascending=False).reset_index(drop=True)

print("------------------------------------------------")
print("AUDITORÍA DE COEFICIENTES (REGRESIÓN LOGÍSTICA)")
print("------------------------------------------------")
print(df_auditoria_pesos)

------------------------------------------------
AUDITORÍA DE COEFICIENTES (REGRESIÓN LOGÍSTICA)
------------------------------------------------
  Característica     Peso_W  Magnitud_Absoluta
0              B -15.848565          15.848565
1              C   2.164877           2.164877
2        Valor_7   1.041914           1.041914
3              E   0.800146           0.800146
4        Valor_6  -0.482270           0.482270
5       Valor_10  -0.301256           0.301256
6              A   0.273226           0.273226
7        Valor_9  -0.110028           0.110028
8        Valor_4  -0.105602           0.105602
9        Valor_3  -0.042590           0.042590


**Revisión de Coeficientes**

La revisión de los coeficientes muestra que la variable `B` (conteo absoluto de palabras negativas) destaca de forma anómala, con un peso de -15.85. En la práctica, esto significa que el modelo ha aprendido que “cuantas más palabras negativas aparecen en un texto, menor es la probabilidad de que sea un mensaje de odio”, lo cual resulta claramente contradictorio.

**Interpretación:** Al revisar manualmente el corpus se observa que los textos de la Clase 0 (No Odio) suelen ser más largos, mientras que los de la Clase 1 (Odio) tienden a ser mensajes breves, muchas veces insultos directos. Como `B` es un conteo absoluto, está reflejando la longitud del documento, no su carga emocional.

**Decisión:**  
- Mantener esta variable introduce un riesgo claro de fuga de información y hace que el modelo sea poco fiable. Por ejemplo, un **mensaje de odio largo** podría clasificarse erróneamente como “No Odio” debido al peso negativo exagerado de `B`. Por este motivo, se decide eliminar la variable del *pipeline* (renunciar a la métrca artificialmente alta del 0.99) para obligar al modelo a apoyarse en señales semánticas reales y enriquecer los criterios de decisión del modelo.
- También se eliminarán las variables derivadas de `B`. Aunque podría resultar interesante analizar su impacto suavizado en un modelo lineal mediante la interacción con el resto de variables, en los modelos basados en árboles dicho efecto se revierte: la fuerte capacidad discriminativa de `B` vuelve a dominar y termina contaminando el modelo.
- Entonces, a parte de la variable `B`, se eliminarán también: `Valor_1`, `Valor_5`, `Valor_6` y `Valor_7`.
- **Consideración:** Se optó por eliminar la variable `Valor_8` y se mantuvo `Valor_6` debido a la fuerte colinealidad entre ellas --> se cambia la decisión y se procede a eliminar `Valor_6`y mantener `Valor_8`.

#### III. Iteración 2: Reentrenamiento con Espacio de Características Actualizado

In [24]:
# 1. Actualizamos la lista de características numéricas (sin 'B', sin sus derivadas y añadiendo 'Valor_8')
COLS_NUMERICAS_V2 = ['A', 'C', 'Valor_3', 'Valor_4', 'Valor_8', 'Valor_9', 'Valor_10'] 

COLS_DROP_V2 = ['label', 'B', 'D', 'Valor_1', 'Valor_2', 'Valor_5', 'Valor_6', 'Valor_7']

# 2. Purgamos la variable seleccionada
X_train_v2 = X_train.drop(columns=COLS_DROP_V2, errors='ignore')
X_test_v2  = X_test.drop(columns=COLS_DROP_V2, errors='ignore')


# 3. Reensamblamos el ColumnTransformer con la nueva lista
preprocesador_v2 = ColumnTransformer(
    transformers=[
        ('num_transforms', num_pipeline, COLS_NUMERICAS_V2),
        ('cat_ordinal', 'passthrough', COLS_ORDINALES)
    ],
    remainder='drop'
)

# 4. Reensamblamos el Pipeline final
pipeline_lr_v2 = Pipeline(steps=[
    ('preprocesador', preprocesador_v2),
    ('modelo', modelo_base) # Usamos la misma configuración de LogisticRegression
])

pipeline_lr_v2

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_transforms', ...), ('cat_ordinal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diff

In [25]:
# 5. Ejecutamos la Validación Cruzada y auditamos los resultados
cv_results_v2 = auditar_modelo(
    pipeline=pipeline_lr_v2,
    X=X_train_v2,
    y=y_train,
    cv=skf,
    metricas=metricas,
    nombre_modelo="Baseline ElasticNet (Post-Purga)"
)

Entrenando Baseline ElasticNet (Post-Purga) con Validación Cruzada...

-------------------------------
MÉTRICAS DE VALIDACIÓN CRUZADA: BASELINE ELASTICNET (POST-PURGA)
-------------------------------
- ROC-AUC  : 0.9695 (+/- 0.0022)
- Accuracy : 0.8938 (+/- 0.0180)
- F1-Macro : 0.8933 (+/- 0.0185)

--------------------------------------
AUDITORÍA DE SOBREAJUSTE (TRAIN vs VAL)
--------------------------------------
- Accuracy Gap : Train (0.8953) vs Val (0.8938) -> Diferencia: 0.0015
- ROC-AUC Gap  : Train (0.9699) vs Val (0.9695) -> Diferencia: 0.0005


Las métricas bajan ligeramente respecto a la versión pre-purga al eliminar la fuga de información (variable B y derivadas). Aún así, el rendimiento es sólido y **realista**: *ROC-AUC* de 0.9695 y *F1-Macro* de 0.8933. Esto demuestra una fuerte separabilidad lineal y establece un baseline estricto (*Accuracy* 89.38%) para comparar con el modelo de árboles.

#### IV. Iteración 2: Auditoría de coeficientes v2

In [26]:
import pandas as pd
import numpy as np

# 1. Ajustamos el pipeline V2 con todo el conjunto de entrenamiento purgado
pipeline_lr_v2.fit(X_train_v2, y_train)

# 2. Extraemos el modelo entrenado
modelo_entrenado_v2 = pipeline_lr_v2.named_steps['modelo']

# 3. Extraemos los nuevos coeficientes
pesos_w_v2 = modelo_entrenado_v2.coef_[0]

# 4. Reconstruimos los nombres (usando las constantes corregidas)
nombres_columnas_v2 = COLS_NUMERICAS_V2 + COLS_ORDINALES

# 5. Creamos el DataFrame de auditoría V2
df_auditoria_pesos_v2 = pd.DataFrame({
    'Caracteristica': nombres_columnas_v2,
    'Peso_W': pesos_w_v2,
    'Magnitud_Absoluta': np.abs(pesos_w_v2)
})

# Ordenamos
df_auditoria_pesos_v2 = df_auditoria_pesos_v2.sort_values(by='Magnitud_Absoluta', ascending=False).reset_index(drop=True)

print("----------------------------------------------")
print("AUDITORÍA DE COEFICIENTES V2 (ESPACIO PURGADO)")
print("----------------------------------------------")
print(df_auditoria_pesos_v2)

----------------------------------------------
AUDITORÍA DE COEFICIENTES V2 (ESPACIO PURGADO)
----------------------------------------------
  Caracteristica    Peso_W  Magnitud_Absoluta
0              C -3.053486           3.053486
1              E  1.010916           1.010916
2       Valor_10 -0.536792           0.536792
3              A -0.360436           0.360436
4        Valor_8  0.340531           0.340531
5        Valor_9  0.317316           0.317316
6        Valor_4  0.167862           0.167862
7        Valor_3 -0.096666           0.096666


**CONCLUSIÓN DE LA FASE 2**

Tras actualizar el conjunto de variables numéricas (eliminar la variable `B`, `Valor_1`, `Valor_6`, `Valor_7` y añadir `Valor_8`), que introducían una fuga de información relacionada con la longitud del texto, el modelo ajusta su *ROC‑AUC* de 0.9944 a 0.9695. Supone una redcución de la métrica a un valor más coherente y se ha conseguido un mayor enriquecimiento del criterio del modelo a la hora de descriminar. 

**1. Rendimiento equilibrado entre clases:**  
La coincidencia casi exacta entre el *Accuracy* y el *F1‑Macro* (ambos en 0.893) indica que el modelo trata por igual a la clase mayoritaria y a la minoritaria. El error se distribuye de forma uniforme, sin favorecer a una clase sobre la otra.

**2. Coeficientes más saludables (Auditoría V2):**  
El nuevo vector de pesos muestra que el modelo ya no depende de una única variable. La magnitud máxima, que antes alcanzaba 15.84, se reduce a un valor mucho más razonable de -3.05 (`C`). Además, la carga predictiva se reparte entre varias características (`C`, `E`, `Valor_10`, `A`, etc.), lo que demuestra que la decisión se construye a partir de señales más ricas y representativas del espacio de características.

**Veredicto:**  
La Regresión Logística con ElasticNet se consolida como un *baseline* sólido con un rendimiento muy alto.

### **2.3. Gradient Boosting: LightGBM**

Repaso de las variables que participan en el modelado:

In [27]:
X_train_v2.head(5)

,A,C,E,Valor_3,Valor_4,Valor_8,Valor_9,Valor_10
9269,0,3,0,0.119973,0.152955,-0.141680,-0.180630,0.221244
5921,0,1,3,0.119973,-0.266832,0.068328,-0.151968,-0.385961
9428,6,12,0,-2.553328,-3.255265,-1.086717,-1.385467,0.221244
6460,0,3,6,0.119973,-0.686619,-0.141680,0.810849,-0.993166
7190,0,4,0,0.119973,0.152955,-0.246684,-0.314500,0.221244


In [28]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

# 1. Configuración del Modelo de Gradiente Boosting
modelo_lgbm = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)

# 2. Ensamble del Pipeline
# Reutilizamos el mismo preprocesador_v2. 
# Aunque los árboles son invariantes a la escala (no necesitan el RobustScaler),
# usar el mismo preprocesador garantiza una comparación justa entre los modelos.
pipeline_lgbm = Pipeline(steps=[
    ('preprocesador', preprocesador_v2),
    ('modelo', modelo_lgbm)
])

pipeline_lgbm

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_transforms', ...), ('cat_ordinal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diff

In [29]:
# 3. Ejecutamos la Validación Cruzada y auditamos los resultados
cv_results_lgbm = auditar_modelo(
    pipeline=pipeline_lgbm,
    X=X_train_v2,
    y=y_train,
    cv=skf,
    metricas=metricas,
    nombre_modelo="LightGBM"
)

Entrenando LightGBM con Validación Cruzada...

-------------------------------
MÉTRICAS DE VALIDACIÓN CRUZADA: LIGHTGBM
-------------------------------
- ROC-AUC  : 0.9759 (+/- 0.0028)
- Accuracy : 0.9193 (+/- 0.0064)
- F1-Macro : 0.9189 (+/- 0.0064)

--------------------------------------
AUDITORÍA DE SOBREAJUSTE (TRAIN vs VAL)
--------------------------------------
- Accuracy Gap : Train (0.9289) vs Val (0.9193) -> Diferencia: 0.0096
- ROC-AUC Gap  : Train (0.9841) vs Val (0.9759) -> Diferencia: 0.0082


**Decisión**: Aunque la variable `E` muestra un comportamiento ordinal, se decide no usar `monotone_constraints`. Forzar una relación estrictamente creciente limitaría la flexibilidad del modelo para captar interacciones más complejas y podría introducir sesgos al impedir que aparezcan excepciones naturales en los datos.

### <span style="color:#2F7597">**2.4. Conclusiones del Modelado**</span>

**Comparativa Rápida: ElasticNet vs LightGBM**

| Métrica     | ElasticNet | LightGBM | Mejora |
|-------------|------------|----------|--------|
| ROC-AUC     | 0.9695     | 0.9759   |  +0.0064 |
| Accuracy    | 0.8938     | 0.9193   |  +0.0255 |
| F1-Macro    | 0.8933     | 0.9189   |  +0.0256 |
| Gap AUC     | 0.0005     | 0.0082   | — |
| Gap Acc     | 0.0015     | 0.0096   | — |

**Conclusiones:**

- **LightGBM supera al baseline lineal** en todas las métricas importantes.  
- El salto en **Accuracy** y **F1-Macro** indica una mejora en la calidad de clasificación.  
- El ligero aumento del gap entre Train y Validación es **normal en modelos no lineales** y sigue dentro de márgenes seguros.  

En conjunto, el cambio supone un trade-off claro: ElasticNet ofrece un comportamiento muy estable y prácticamente sin sobreajuste, pero LightGBM aporta un salto real en capacidad predictiva gracias a su flexibilidad para capturar relaciones no lineales. Aceptamos un ligero aumento en el gap entre entrenamiento y validación porque el rendimiento adicional compensa con creces ese coste. En este punto, **LightGBM se posiciona como la opción más sólida para avanzar**.